In [65]:
# based on https://github.com/OSeMOSYS/CLEWs/blob/master/tools/visualisation/CBC_results.ipynb
import pandas as pd
import os, sys
import plotly.express as px
import ast
from collections import defaultdict
import yaml
from pathlib import Path

In [66]:
root_folder = Path.cwd().parent
yaml_file = root_folder / "config/clews_config/clewsy_template.yaml"
with open(yaml_file, "r") as f:
        configdata = yaml.load(f, Loader=yaml.SafeLoader)
country_name = configdata['Model']
data_file = root_folder / f"results/{country_name}/data.txt"

folder = root_folder / f"results/{country_name}/results"

dfs = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        name = os.path.splitext(file)[0]   # remove .csv
        dfs[name] = pd.read_csv(os.path.join(folder, file))

In [67]:
year_list = configdata['Years']
years = pd.Series(([int(i) for i in year_list])).drop_duplicates()

name_color_codes = pd.read_csv(('name_color_codes.csv'), encoding='latin-1')

det_col = dict([(c,n) for c,n in zip(name_color_codes.code, name_color_codes.name_english)])
color_dict = dict([(n,c) for n,c in zip(name_color_codes.name_english, name_color_codes.colour)])

In [68]:
# Land use names
with open(f"{root_folder}/results/Philippines/CROP.csv") as f:
    crops = {line.strip() for line in f.readlines()[1:]}

modelist_path = f"{root_folder}/workflow/submodules/clewsy/optn_mds.txt"
with open(modelist_path, 'r') as f:
    text = f.read()

# Remove quotes, spaces, and trailing commas, then split into a list
mode_list = [item.strip().strip("'").strip('"') for item in text.split(',') if item.strip()]

mode_crop_combo = dict([(i+1, m) for i, m in enumerate(mode_list)])

water_supply = {'I':'Irrigated',
                'REGION':'Rain-fed'}

input_level = {'L':'Low',
               'H':'High'}

land_use_map = {
    'Bar': 'Barren',
    'Bui': 'Built-up',
    'Wat': 'Water bodies',
    'For': 'Forest',
    'Gra': 'Grassland and Woodland',
    'AGR': 'Agriculture'
}

In [69]:
def df_filter(df,lb,ub,t_exclude):
    df['TECHNOLOGY'] = df['TECHNOLOGY'].str[lb:ub]
    df['VALUE'] = df['VALUE'].astype('float64')
    df = df[~df['TECHNOLOGY'].isin(t_exclude)].pivot_table(index='YEAR', 
                                          columns='TECHNOLOGY',
                                          values='VALUE', 
                                          aggfunc='sum').reset_index().fillna(0)
    df = df.reindex(sorted(df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)
    new_df = pd.DataFrame()
    new_df['YEAR'] = years
    new_df['YEAR'] = new_df['YEAR'].astype(int)
    df['YEAR'] = df['YEAR'].astype(int)
    new_df = pd.merge(new_df,df, how='outer', on='YEAR').fillna(0)
    return new_df

In [70]:
def df_plot(df, y_label, title):
    """
    Plot a stacked bar chart from a DataFrame using Plotly Express.

    Parameters:
    - df: pandas DataFrame with a 'YEAR' column and other columns as series to plot
    - y_label: string, label for the y-axis
    - title: string, plot title
    """
    # Copy the dataframe to avoid modifying original
    df = df.copy()
    
    # Only columns that exist in color_dict
    value_cols = [c for c in df.columns if c != 'YEAR']

    # Melt the DataFrame for Plotly Express
    df_melt = df.melt(
        id_vars='YEAR', 
        value_vars=value_cols,
        var_name='Technology', 
        value_name='VALUE'   
    )

    # Create stacked bar chart
    fig = px.bar(
        df_melt,
        x='YEAR',
        y='VALUE',
        color='Technology',
        color_discrete_map=color_dict,
        labels={
            'VALUE': y_label,  # must match value_name
            'YEAR': 'Year'     # x-axis
        },
        title=title,
    )

    fig.update_layout(barmode='relative')
    fig.show()

In [71]:
def df_line_plot(df, y_label, title):
    """
    Plot a multi-series line chart from a DataFrame using Plotly Express.

    Parameters:
    - df: pandas DataFrame with a 'YEAR' column and other columns as series
    - y_label: string, label for the y-axis
    - title: string, plot title
    """
    df = df.copy()

    # All columns except YEAR
    value_cols = [c for c in df.columns if c != 'YEAR']

    # Melt for Plotly
    df_melt = df.melt(
        id_vars='YEAR',
        value_vars=value_cols,
        var_name='Technology',
        value_name='Value'
    )

    # Plot
    fig = px.line(
        df_melt,
        x='YEAR',
        y='Value',
        color='Technology',
        color_discrete_map=color_dict,
        markers=True,
        labels={
        'Value': y_label,  # y-axis
        'YEAR': 'Year'     # x-axis
         },
        title=title,
    )

    fig.show()

Energy

In [ ]:
#Power generation (Detailed)
gen_df = dfs['ProductionByTechnologyAnnual'][(dfs['ProductionByTechnologyAnnual'].TECHNOLOGY.str.startswith('PWR') |
                                                     dfs['ProductionByTechnologyAnnual'].TECHNOLOGY.str.startswith('IMP')) & 
                                                   dfs['ProductionByTechnologyAnnual'].FUEL.str.startswith('ELC')].drop('REGION', axis=1)

gen_df = df_filter(gen_df,3,6,['TRN'])

ele_exp_df = dfs['TotalTechnologyAnnualActivity'][dfs['TotalTechnologyAnnualActivity'].TECHNOLOGY.str.startswith('EXPELC')].drop('REGION', axis=1)

if not ele_exp_df.empty:
    ele_exp_df = df_filter(ele_exp_df,3,6,['TRN']).rename(columns={'Electricity':'Electricity exports'})
    gen_df = gen_df.merge(ele_exp_df)

if 'Electricity' in gen_df.columns:
    gen_df['Net electricity imports'] = gen_df['Electricity'] - gen_df['Electricity exports']
    gen_df.drop(['Electricity','Electricity exports'], axis=1, inplace=True)

df_plot(gen_df, 'Petajoules (PJ)', 'Power Generation')

In [ ]:
# Power generation capacity (detailed)
cap_df = dfs['TotalCapacityAnnual'][dfs['TotalCapacityAnnual'].TECHNOLOGY.str.startswith('PWR')].drop('REGION', axis=1)
cap_df = df_filter(cap_df,3,6,['CNT','TRN','CST','CEN','SOU','NOR']) #added in CCG, OCG, SPV
df_plot(cap_df,'Gigawatts (GW)','Power Generation Capacity')

In [74]:
# Fuel use for power generation
gen_use_df = dfs['TotalTechnologyAnnualActivity'][dfs['TotalTechnologyAnnualActivity'].TECHNOLOGY.str.startswith('RNW') | 
                                                      dfs['TotalTechnologyAnnualActivity'].TECHNOLOGY.str.startswith('MIN')].drop('REGION', axis=1)

gen_use_df = df_filter(gen_use_df,3,6,['HYD','SPV','WND', 'SUR', 'SOL','BIO','LND','PRC','WOF','WON','GEO'])
df_plot(gen_use_df,'Petajoules (PJ)','Power Generation (Fuel use)')

Climate

In [75]:
#Power sector emissions
emissions_df = dfs['AnnualTechnologyEmission'][dfs['AnnualTechnologyEmission'].TECHNOLOGY.str.startswith('DEM') | dfs['AnnualTechnologyEmission'].TECHNOLOGY.str.startswith('MIN') | dfs['AnnualTechnologyEmission'].TECHNOLOGY.str.startswith('PWR')].drop('REGION', axis=1)
# emissions_df['TECHNOLOGY'] = emissions_df['TECHNOLOGY'].str.replace('CCG', 'NGS', regex=True)
# emissions_df['TECHNOLOGY'] = emissions_df['TECHNOLOGY'].str.replace('OCG', 'NGS', regex=True)
emissions_fuel_df = df_filter(emissions_df,3,6,[])
df_plot(emissions_fuel_df,'Million tonnes of CO2','CO2 emissions by source')

Land

In [76]:
crops_total_df = dfs['TotalAnnualTechnologyActivityByMode'][dfs['TotalAnnualTechnologyActivityByMode'].TECHNOLOGY.str.startswith('LNDAGR')].drop('REGION', axis=1)
#crops_total_df['land_use'] = crops_total_df.m.map(mode_search)

crops_total_df['MODE_OF_OPERATION'] = crops_total_df['MODE_OF_OPERATION'].astype(int)
crops_total_df['crop_combo'] = crops_total_df['MODE_OF_OPERATION'].map(mode_crop_combo)
crops_total_df['land_use'] = crops_total_df['crop_combo'].str[0:3]
crops_total_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)

crops_total_df = crops_total_df[crops_total_df['land_use'].str.startswith(tuple(crops))]
crops_total_df = crops_total_df.pivot_table(index='YEAR', 
                                            columns='land_use',
                                            values='VALUE', 
                                            aggfunc='sum').reset_index().fillna(0)
crops_total_df = crops_total_df.reindex(sorted(crops_total_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col).astype('float64')

if not crops_total_df.empty:
    df_plot(crops_total_df,'Land area (1000 sq.km.)','Area by crop')

In [77]:
land_total_df = dfs['TotalAnnualTechnologyActivityByMode'][
    dfs['TotalAnnualTechnologyActivityByMode'].TECHNOLOGY.str.startswith('LNDAGR')
].drop('REGION', axis=1)

land_total_df['MODE_OF_OPERATION'] = land_total_df['MODE_OF_OPERATION'].astype(int)
land_total_df['crop_combo'] = land_total_df['MODE_OF_OPERATION'].map(mode_crop_combo)

# Keep short codes for grouping
land_total_df['land_use'] = land_total_df['crop_combo'].str[0:3]

land_total_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)

land_total_df = land_total_df.pivot_table(
    index='YEAR', 
    columns='land_use',
    values='VALUE', 
    aggfunc='sum'
).reset_index()

# Aggregate crops into AGR
land_total_df['AGR'] = 0
for crop in crops:
    if crop in land_total_df.columns:
        land_total_df['AGR'] += land_total_df[crop]
        land_total_df.drop(crop, axis=1, inplace=True)

land_total_df = land_total_df.rename(columns=land_use_map)

land_total_df = (
    land_total_df
    .reindex(sorted(land_total_df.columns), axis=1)
    .set_index('YEAR')
    .reset_index()
    .rename(columns=det_col)
    .astype('float64')
)

if not land_total_df.empty:
    df_plot(land_total_df, 'Land area (1000 sq.km.)', 'Area by land cover type')

In [ ]:
## Area by land cover type for each Cluster

# land_total_df = dfs['TotalAnnualTechnologyActivityByMode'][
#     dfs['TotalAnnualTechnologyActivityByMode'].TECHNOLOGY.str.startswith('LNDAGR')
# ].drop('REGION', axis=1)

# # Extract cluster (last character of TECHNOLOGY)
# land_total_df['cluster'] = land_total_df['TECHNOLOGY'].str[-1].astype(int)

# for cluster_id in sorted(land_total_df['cluster'].unique()):
    
#     df_cluster = land_total_df[land_total_df['cluster'] == cluster_id].copy()

#     df_cluster['MODE_OF_OPERATION'] = df_cluster['MODE_OF_OPERATION'].astype(int)
#     df_cluster['crop_combo'] = df_cluster['MODE_OF_OPERATION'].map(mode_crop_combo)

#     # Keep short codes for grouping
#     df_cluster['land_use'] = df_cluster['crop_combo'].str[0:3]

#     df_cluster.drop(['MODE_OF_OPERATION','crop_combo','cluster'], axis=1, inplace=True)
    
#     df_cluster = df_cluster.pivot_table(
#         index='YEAR', 
#         columns='land_use',
#         values='VALUE', 
#         aggfunc='sum'
#     ).reset_index().fillna(0)

#     # Aggregate crops into AGR
#     # df_cluster['AGR'] = 0
#     # for crop in crops:
#     #     if crop in df_cluster.columns:
#     #         df_cluster['AGR'] += df_cluster[crop]
#     #         df_cluster.drop(crop, axis=1, inplace=True)

#     df_cluster = df_cluster.rename(columns=land_use_map)
#     # df_clustertotal = df_cluster.copy()
#     # df_clustertotal['TOTAL'] = df_clustertotal.drop(columns='YEAR').sum(axis=1)
#     # print(df_clustertotal['TOTAL'])
#     df_cluster = (
#         df_cluster
#         .reindex(sorted(df_cluster.columns), axis=1)
#         .set_index('YEAR')
#         .reset_index()
#         .rename(columns=det_col)
#         .astype('float64')
#     )

#     if not df_cluster.empty:
#         df_plot(
#             df_cluster, 
#             'Land area (1000 sq.km.)', 
#             f'Area by land cover type (Cluster {cluster_id})'
#         )


In [79]:
for each in water_supply.keys():
    crops_ws_df = dfs['TotalAnnualTechnologyActivityByMode'][dfs['TotalAnnualTechnologyActivityByMode'].TECHNOLOGY.str.startswith('LNDAGR')].drop('REGION', axis=1)
    crops_ws_df['MODE_OF_OPERATION'] = crops_ws_df['MODE_OF_OPERATION'].astype(int)
    crops_ws_df['crop_combo'] = crops_ws_df['MODE_OF_OPERATION'].map(mode_crop_combo)
    crops_ws_df = crops_ws_df[(crops_ws_df.crop_combo.str.startswith(tuple(crops))) & (crops_ws_df.crop_combo.str[4:5] == each)]
    crops_ws_df['land_use'] = crops_ws_df['crop_combo'].str[0:3]
    crops_ws_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)
    
    crops_ws_df = crops_ws_df.pivot_table(index='YEAR', 
                                          columns='land_use',
                                          values='VALUE', 
                                          aggfunc='sum').reset_index().fillna(0)
    crops_ws_df = crops_ws_df.reindex(sorted(crops_ws_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)
    if len(crops_ws_df.columns) > 1:
        df_plot(crops_ws_df,'Land area (1000 sq.km.)','Area by crop (' + water_supply[each] + ')')

In [80]:
land_use = ['HI', 'HR', 'LI', 'LR']
for cp in crops:
        crops_agr_df = dfs['TotalAnnualTechnologyActivityByMode'][dfs['TotalAnnualTechnologyActivityByMode'].TECHNOLOGY.str.startswith('LNDAGR')].drop('REGION', axis=1)
        crops_agr_df['MODE_OF_OPERATION'] = crops_agr_df['MODE_OF_OPERATION'].astype(int)
        crops_agr_df['crop_combo'] = crops_agr_df['MODE_OF_OPERATION'].map(mode_crop_combo)
        crops_agr_df = crops_agr_df[(crops_agr_df.crop_combo.str.startswith(cp))]
        crops_agr_df['land_use'] = crops_agr_df['crop_combo'].str[3:5]
        crops_agr_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)
        crops_agr_df = crops_agr_df.pivot_table(index='YEAR', 
                                              columns='land_use',
                                              values='VALUE', 
                                              aggfunc='sum').reset_index().fillna(0)
        crops_agr_df = crops_agr_df.reindex(sorted(crops_agr_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)
        if len(crops_agr_df.columns) > 1:
            df_plot(crops_agr_df,'Land area (1000 sq.km.)', ''+ cp + ' '+'Crop Area by Land Use')
    

In [81]:
crops_prod_df = dfs['ProductionByTechnologyAnnual'][dfs['ProductionByTechnologyAnnual'].FUEL.str.startswith('CRP')].drop('REGION', axis=1)
crops_prod_df['FUEL'] = crops_prod_df['FUEL'].str[3:7]
crops_prod_df['VALUE'] = crops_prod_df['VALUE'].astype('float64')

crops_prod_df = crops_prod_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)
crops_prod_df = crops_prod_df.reindex(sorted(crops_prod_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)
crops_prod_df['YEAR'] = years

if len(crops_prod_df.columns) > 1:
    df_plot(crops_prod_df,'Production (Million tonnes)','Crop production')

In [82]:
crops_yield_df = crops_prod_df / crops_total_df
crops_yield_df['YEAR'] = years

# Multiply all non-YEAR columns
crops_yield_df.loc[:, crops_yield_df.columns != 'YEAR'] *= 10
#crops_yield_df = crops_yield_df[crops_yield_df['YEAR'] >= 2035]
if not crops_yield_df.empty:
    df_line_plot(
        crops_yield_df,
        'Yield (t/ha)',
        'Yield (tonnes/hectare)'
    )



Water Sector

In [83]:
wat_dem_df = dfs['ProductionByTechnologyAnnual'][dfs['ProductionByTechnologyAnnual'].FUEL.str[0:6].isin(['AGRWAT','PUBWAT','PWRWAT'])].drop('REGION', axis=1)
#wat_dem_df.to_csv('df_waterdemand_nosw.csv', index=False)
wat_dem_df['FUEL'] = wat_dem_df['FUEL'].str[0:3]
wat_dem_df['VALUE'] = wat_dem_df['VALUE'].astype('float64')
wat_dem_df = wat_dem_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)

pwrwat_df = dfs['ProductionByTechnologyAnnual'][(dfs['ProductionByTechnologyAnnual'].TECHNOLOGY.str.startswith('PWR')) & 
                                                   dfs['ProductionByTechnologyAnnual'].FUEL.str.startswith('WTR')].drop('REGION', axis=1)
pwrwat_df['FUEL'] = pwrwat_df['FUEL'].str[0:3]
pwrwat_df['VALUE'] = pwrwat_df['VALUE'].astype('float64')

pwrwat_df = pwrwat_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)

if not pwrwat_df.empty:
    wat_dem_df['PWR_WAT'] = wat_dem_df['PWR'] - pwrwat_df['WTR']
    
wat_dem_df = wat_dem_df.reindex(sorted(wat_dem_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)

if 'Power sector' in wat_dem_df.columns:
    wat_dem_df.rename(columns={'Power sector':'Power sector (withdrawal)'}, inplace=True)

wat_dem_df = wat_dem_df.loc[:, (wat_dem_df != 0).any(axis=0)]

df_plot(wat_dem_df, 'Billion m3', 'Water Demand')

In [84]:
wat_bal_df = dfs['ProductionByTechnologyAnnual'][dfs['ProductionByTechnologyAnnual'].FUEL.str.startswith('WTR')].drop('REGION', axis=1)
wat_bal_df['FUEL'] = wat_bal_df['FUEL'].str[3:6]
wat_bal_df['VALUE'] = wat_bal_df['VALUE'].astype('float64')
wat_bal_df = wat_bal_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)
wat_bal_df = wat_bal_df.reindex(sorted(wat_bal_df.columns), axis=1).set_index('YEAR').reset_index().rename(columns=det_col)
wat_bal_df['Irrigation'] = wat_dem_df['Agriculture']
wat_bal_df['YEAR'] = years
for each in wat_bal_df.columns:
    if each in ['Evapotranspiration', 'Groundwater recharge', 'Surface water run-off']:
        wat_bal_df[each] = wat_bal_df[each].mul(-1)

if 'Power sector (withdrawal)' in wat_bal_df:
    wat_bal_df['Surface water run-off'] = wat_bal_df['Surface water run-off'] + wat_dem_df['Power sector (withdrawal)'] - wat_dem_df['Power sector (consumptive use)']

df_plot(wat_bal_df, 'Billion m3', 'Water Balance')